# Blog-to-LinkedIn Converter

## 1. Project Overview

**Task:** Rewrite a blog article into LinkedIn post format — with a hook, short paragraphs, professional tone, relevant emojis, and an engagement-driving CTA.

**Approach:** We test multiple prompt strategies (zero-shot, few-shot, role-based) and evaluate output quality with a rubric: hook strength, tone, length compliance, information preservation, and engagement CTA.

**Stack:** `LangChain` + `ChatOllama` + `qwen3.5:9b` — no API keys required.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Transform** long-form content into platform-specific format |
| 2 | **Control tone** — professional but conversational for LinkedIn |
| 3 | **Enforce constraints** — word count, structure, emoji usage |
| 4 | Compare **zero-shot vs few-shot** prompt strategies |
| 5 | Design a **rubric** for evaluating content transformation quality |

## 3. Problem Statement

Blog articles are 800–2,000+ words. LinkedIn posts perform best at 150–300 words with a specific structure: attention-grabbing first line, short paragraphs, strategic white space, and a closing question.

## 4. Why This Matters

- **Content repurposing** is how modern creators maximize ROI on writing
- **Platform-aware formatting** is a transferable prompt engineering skill
- **Tone control** teaches LLM instruction design for real business use

## 5. Setup

In [ ]:
import os, json, re, time, warnings
from textwrap import dedent
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

LLM_MODEL = "qwen3.5:9b"
llm = ChatOllama(model=LLM_MODEL, temperature=0.7)  # Slightly creative

def clean(text):
    if "<think>" in text: text = text.split("</think>")[-1].strip()
    return text.strip()

def ask(system, user):
    return clean(llm.invoke([SystemMessage(content=system), HumanMessage(content=user)]).content)

print(f"LLM ready: {ask('Reply with one word.', 'Say ready.')[:20]}")

## 6. Sample Blog Articles

In [ ]:
BLOGS = {
    "ml_trends": {
        "title": "Machine Learning in 2026: Five Trends Reshaping the Industry",
        "article": dedent("""The machine learning landscape has shifted dramatically in the past year. What was cutting-edge in 2024 is now table stakes. Here are five trends reshaping how companies build and deploy ML systems.

First, foundation models have become commoditized. Multiple providers now offer comparable quality, driving prices down 80% in 18 months. The competitive moat has shifted from model quality to data quality and fine-tuning expertise.

Second, RAG (Retrieval-Augmented Generation) has become the default architecture for enterprise AI. Instead of fine-tuning models on proprietary data, companies retrieve relevant context at inference time. This reduces costs, improves accuracy, and makes updates instant.

Third, evaluation frameworks have finally caught up. Tools like RAGAS, DeepEval, and custom LLM-as-judge pipelines allow systematic testing of AI outputs. Companies that don't measure can't improve.

Fourth, small models are surprisingly competitive. For focused tasks like classification, extraction, and routing, models under 10B parameters often match larger models at a fraction of the cost and latency.

Fifth, the "AI engineer" role has emerged. These are software engineers who specialize in building AI-powered products, bridging the gap between ML research and production systems. They focus on prompting, orchestration, and evaluation rather than training models from scratch.

The bottom line: ML is no longer just for researchers. Product teams are building AI features directly, and the tools are becoming accessible to developers without PhD-level ML expertise.""")
    },
    "remote_work": {
        "title": "The Hidden Cost of Always-On Culture in Remote Teams",
        "article": dedent("""Remote work promised freedom. For many, it delivered the opposite: an always-on culture where the laptop never closes and the Slack notifications never stop.

A recent study by the Harvard Business Review found that remote workers log an average of 2.5 extra hours per day compared to their in-office counterparts. More concerning, 67% of remote workers report checking messages outside of working hours daily, and 43% say they feel guilty when they're not available.

The cost is real. Burnout rates among remote workers have increased 38% since 2020. Creativity and deep work suffer when every moment is interruptible. And the irony is that productivity often decreases, not increases, when people are always available.

The best remote-first companies have solved this by being intentional about async communication. GitLab documents everything. Basecamp bans real-time chat for most communication. Doist tracks maker time versus manager time.

Three practical fixes: First, establish core hours (4-5 hours of overlap) and protect the rest. Second, default to async — if it can be a document or video, don't make it a meeting. Third, model the behavior from the top — leaders who respond at midnight create cultures of midnight responses.

Remote work is better when it's actually remote, not just office work relocated to your living room.""")
    },
}

for key, blog in BLOGS.items():
    print(f"{key}: {blog['title']} | {len(blog['article'].split())} words")

---

## 7. Approach 1: Zero-Shot Conversion

In [ ]:
ZERO_SHOT_SYSTEM = """Convert this blog post into a LinkedIn post.
Requirements:
- Start with a compelling hook (first line grabs attention)
- Use short paragraphs (1-2 sentences each)
- Add line breaks between paragraphs for readability
- Use emojis sparingly (2-4 total)
- End with a question to drive engagement
- Keep under 250 words
- Professional but conversational tone
- Return ONLY the LinkedIn post"""

for key, blog in BLOGS.items():
    post = ask(ZERO_SHOT_SYSTEM, f"Blog:\n{blog['article']}")
    print(f"\n{'=' * 60}")
    print(f"ZERO-SHOT: {blog['title']}")
    print("=" * 60)
    print(post)
    print(f"\nWord count: {len(post.split())}")

## 8. Approach 2: Few-Shot with Examples

In [ ]:
FEW_SHOT_SYSTEM = """Convert blog posts into LinkedIn posts. Here's the style to follow:

EXAMPLE INPUT: "Why Documentation Matters: A blog about how good docs reduce onboarding time by 50%..."

EXAMPLE OUTPUT:
"I used to think documentation was a waste of time.

Then I joined a team with zero docs.

It took me 3 weeks to understand the codebase. My colleague who joined a well-documented team? 3 days.

Good documentation isn't overhead. It's a multiplier.

\ud83d\udcdd Here's what actually works:
\u2022 READMEs that answer \"how do I get started?\" in 5 minutes
\u2022 Architecture decision records (ADRs) for the \"why\"
\u2022 Runbooks for the 3am pages

The best teams I've worked with treat docs like code: reviewed, tested, and kept current.

What's the biggest documentation gap you've seen? \ud83d\udc47"

Now convert the blog below using the same style. Under 250 words. Professional but conversational."""

blog = BLOGS["ml_trends"]
post = ask(FEW_SHOT_SYSTEM, f"Blog title: {blog['title']}\n\n{blog['article']}")
print("FEW-SHOT RESULT")
print("=" * 60)
print(post)
print(f"\nWord count: {len(post.split())}")

## 9. Approach 3: Role-Based Prompting

In [ ]:
ROLE_SYSTEM = """You are a LinkedIn content strategist with 500K followers who specializes in 
turning long-form content into viral LinkedIn posts.

Your signature style:
- Bold opening statement that challenges conventional wisdom
- Personal anecdote or concrete example
- 3-5 key insights as short punchy lines
- Closing with a reflective question
- Exactly 2-3 emojis, never more
- Under 200 words

Convert the blog below. Return ONLY the post."""

blog = BLOGS["remote_work"]
post = ask(ROLE_SYSTEM, f"Blog:\n{blog['article']}")
print("ROLE-BASED RESULT")
print("=" * 60)
print(post)
print(f"\nWord count: {len(post.split())}")

## 10. Quality Evaluation Rubric

In [ ]:
EVAL_SYSTEM = """You evaluate LinkedIn posts converted from blog articles.
Score each criterion 1-5:

1. hook_strength: Does the first line grab attention? (1=boring, 5=can't scroll past)
2. tone: Professional but conversational? (1=too formal/casual, 5=perfect LinkedIn tone)
3. length: Under 250 words? Good use of white space? (1=wall of text, 5=perfect)
4. info_preserved: Key ideas from the original blog kept? (1=lost most, 5=captured all key points)
5. engagement_cta: Ends with something that drives comments? (1=no CTA, 5=great question)

Return JSON: {"hook_strength": N, "tone": N, "length": N, "info_preserved": N, "engagement_cta": N, "total": N, "feedback": "brief note"}"""

def parse_eval(text):
    text = clean(text)
    if "```" in text: text = re.sub(r"```(?:json)?\s*", "", text).replace("```", "")
    s, e = text.find("{"), text.rfind("}") + 1
    if s >= 0 and e > s:
        try: return json.loads(text[s:e])
        except: pass
    return None

# Evaluate all three approaches on the ML trends blog
blog = BLOGS["ml_trends"]
approaches = {
    "Zero-shot": ask(ZERO_SHOT_SYSTEM, f"Blog:\n{blog['article']}"),
    "Few-shot": ask(FEW_SHOT_SYSTEM, f"Blog title: {blog['title']}\n\n{blog['article']}"),
    "Role-based": ask(ROLE_SYSTEM, f"Blog:\n{blog['article']}"),
}

print("QUALITY EVALUATION")
print("=" * 60)
for name, post in approaches.items():
    resp = ask(EVAL_SYSTEM, f"Original blog topic: {blog['title']}\n\nLinkedIn post:\n{post}")
    scores = parse_eval(resp)
    if scores:
        total = sum(scores.get(k, 0) for k in ["hook_strength","tone","length","info_preserved","engagement_cta"])
        print(f"\n{name}: {total}/25")
        for k in ["hook_strength","tone","length","info_preserved","engagement_cta"]:
            print(f"  {k}: {scores.get(k, '?')}/5")
        print(f"  Feedback: {scores.get('feedback', 'N/A')}")

## 11. Audience Variants

The same blog can target different LinkedIn audiences.

In [ ]:
audiences = {
    "Technical IC": "Write for senior software engineers. Use technical language. Focus on implementation details.",
    "Engineering Manager": "Write for engineering managers. Focus on team impact, process, and decision-making.",
    "Founder/CEO": "Write for startup founders. Focus on business impact, ROI, and strategic implications.",
}

blog = BLOGS["ml_trends"]
for audience, instruction in audiences.items():
    sys_prompt = f"{ZERO_SHOT_SYSTEM}\n\nAudience: {instruction}"
    post = ask(sys_prompt, f"Blog:\n{blog['article']}")
    print(f"\n--- {audience} ---")
    print(post[:300])
    print(f"[{len(post.split())} words]\n")

## 12. Common Mistakes & Key Takeaways

| Mistake | Fix |
|---------|-----|
| Copy-pasting blog paragraphs | Rewrite for scanning, not reading |
| Too many emojis | 2-3 max, used for structure not decoration |
| No hook | First line must stop the scroll |
| No CTA | End with a question or call to action |
| Too long | LinkedIn posts over 300 words lose engagement |

| # | Takeaway |
|---|----------|
| 1 | **Few-shot prompting** produces more consistent style than zero-shot |
| 2 | **Role-based prompting** adds personality and platform awareness |
| 3 | **Audience targeting** makes the same content serve different readers |
| 4 | **Rubric-based evaluation** makes quality measurable, not subjective |
| 5 | **The hook is everything** — LinkedIn shows only the first 2 lines before "see more" |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*